# Laboratorio Evaluado: WBCD y Evaluación Multimodelo

**Reto Autónomo: Clínica Oncológica**

Eres el Consultor IA de un hospital. El dataset WBCD contiene radios y perímetros de células. Debes entrenar k-NN y Árboles de Decisión, y validar qué pasa si no escalas los datos.

## 1. Carga de Datos y Exploración

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, recall_score, accuracy_score

# Nombres de columnas del WDBC: id, diagnosis + 10 medidas x (mean, se, worst)
base = ["radius", "texture", "perimeter", "area", "smoothness",
        "compactness", "concavity", "concave_points", "symmetry", "fractal_dimension"]
cols = ["id", "diagnosis"] + [f"{b}_{s}" for s in ["mean", "se", "worst"] for b in base]

try:
    df_wbcd = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data",
                          header=None, names=cols)
except Exception:
    # Respaldo sin internet: mismo dataset WDBC incluido en scikit-learn
    from sklearn.datasets import load_breast_cancer
    d = load_breast_cancer()
    df_wbcd = pd.DataFrame(d.data, columns=cols[2:])
    df_wbcd.insert(0, "diagnosis", np.where(d.target == 0, "M", "B"))  # en sklearn 0 = maligno
    df_wbcd.insert(0, "id", range(len(df_wbcd)))

print(df_wbcd.shape)
print(df_wbcd["diagnosis"].value_counts())
df_wbcd.head()


## 2. Preprocesamiento

In [ ]:
# Diagnosis a binario: M (maligno) = 1, B (benigno) = 0
df_wbcd["diagnosis"] = df_wbcd["diagnosis"].map({"M": 1, "B": 0})

# X (features, sin id ni diagnosis) e y (target)
X = df_wbcd.drop(columns=["id", "diagnosis"])
y = df_wbcd["diagnosis"]

# 80% train / 20% test (stratify mantiene la proporción M/B)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)


## 3. Escalamiento (Solo para k-NN)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # aprende media/desv. SOLO de train
X_test_scaled = scaler.transform(X_test)        # aplica lo aprendido a test (evita data leakage)

# Comparación: escalas muy distintas entre variables (area ~ 600 vs smoothness ~ 0.1)
print(X_train[["area_mean", "smoothness_mean"]].describe().loc[["mean", "std"]])


## 4. Modelado y Matrices de Confusión

In [ ]:
def reporte(nombre, modelo, Xte):
    y_pred = modelo.predict(Xte)
    cm = confusion_matrix(y_test, y_pred)  # filas = real, columnas = predicho
    tn, fp, fn, tp = cm.ravel()
    print(f"--- {nombre} ---")
    print(cm)
    print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
    print(f"Accuracy={accuracy_score(y_test, y_pred):.3f} | Recall maligno={recall_score(y_test, y_pred):.3f}\n")

# k-NN con datos escalados
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)
reporte("k-NN (escalado)", knn, X_test_scaled)

# Árbol de decisión (max_depth=4)
tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
reporte("Árbol (sin escalar)", tree, X_test)

# Validación pedida: ¿qué pasa si NO escalamos?
knn_ns = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
reporte("k-NN (SIN escalar)", knn_ns, X_test)

tree_s = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train_scaled, y_train)
reporte("Árbol (escalado)", tree_s, X_test_scaled)


## Evaluación Teórica / Decisión de Negocios

1. En este escenario médico, ¿qué implica un Error Tipo II (Falso Negativo) y por qué cuesta vidas?
2. ¿Por qué crees que el Árbol de Decisión no falló estrepitosamente incluso si no hubieses escalado los datos?

*Escribe tu respuesta aquí:*

**1. Error Tipo II (Falso Negativo).** Es cuando el modelo clasifica como **benigno (0)** a una paciente que en realidad tiene un tumor **maligno (1)**. En la matriz de confusión es la celda de abajo a la izquierda (FN). Cuesta vidas porque la paciente recibe un "todo está bien", no se le practican más estudios ni se inicia tratamiento, y el cáncer avanza sin ser detectado hasta una etapa donde es más difícil o imposible de tratar. En cambio, un Falso Positivo (Error Tipo I) genera angustia y una biopsia innecesaria, pero la paciente sigue con vida. Por eso en medicina se prioriza el **Recall (sensibilidad)** de la clase maligna sobre el accuracy global. En nuestros resultados: k-NN escalado tuvo 4 FN (recall 0.905), el árbol tuvo 8 FN (recall 0.810) y k-NN sin escalar tuvo 9 FN (recall 0.786). Para la clínica, el mejor modelo es el k-NN escalado, aunque 4 casos perdidos siguen siendo demasiados: se podría bajar el umbral de decisión para reducir FN a costa de más FP.

**2. ¿Por qué el árbol no falla sin escalar?** Porque el árbol no usa distancias: en cada nodo evalúa una sola variable a la vez con una pregunta del tipo "¿perimeter_worst <= 105.9?". Escalar con StandardScaler es una transformación monótona (resta una constante y divide por otra) que no cambia el *orden* de los valores, así que el mejor punto de corte separa exactamente las mismas muestras, solo expresado en otra unidad. Lo comprobamos: el árbol con y sin escalar dio la **misma matriz de confusión** (70/2/8/34). En cambio, k-NN sí depende de la distancia euclidiana: variables con magnitudes grandes (area ~ 650, perimeter ~ 90) dominan sobre las pequeñas (smoothness ~ 0.1), distorsionando quiénes son los "vecinos". Por eso k-NN empeoró sin escalar (de 4 a 9 falsos negativos).
